# Explainable DR Classification — Portable project

Open this notebook from the Medical_Dipatie folder. The original notebook is unchanged.

Run the setup and launch scripts described in `README_PORTABLE.md`. They create a Python 3.11 environment for this machine. Select the Python (Medical Dipatie) kernel.

The first code cell restores the supplied APTOS images from the archives in this folder’s `Project Files` directory if needed. This requires approximately 8.3 GiB of disk space and may take
several minutes; OneDrive may then sync these files. No downloads or training are
performed by this setup step. Keep all five archives until restoration completes.

The demo defaults to CPU, no DataLoader subprocesses, and small XAI samples.
Training is disabled unless you explicitly set `ALLOW_TRAINING = True`.


In [ ]:
%pip install opencv-python-headless==4.11.0.86

In [ ]:
from pathlib import Path
import subprocess
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    requirements_file = candidate / "requirements-windows.txt"
    if requirements_file.is_file():
        break
else:
    raise FileNotFoundError(
        "Open Jupyter from this repository, or run the notebook from a subfolder."
    )

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)])


In [ ]:
%pip install lime==0.2.0.1 scikit-image==0.25.2

In [ ]:
%pip install captum

In [ ]:
from pathlib import Path
import csv
import shutil
import zipfile

# If you launch Jupyter elsewhere, set this to your Medical_Dipatie folder.
PROJECT_ROOT_OVERRIDE = r"" 
RESTORE_DATA_FROM_ARCHIVES = True

def is_project_root(path):
    return (path / "src" / "dr_pipeline.py").is_file() and (path / "configs" / "base.yaml").is_file()

start = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
PROJECT_ROOT = next((p.resolve() for p in (start, *start.parents) if is_project_root(p)), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter in Medical_Dipatie, or set PROJECT_ROOT_OVERRIDE above.")

# Both extracted data and optional archives travel with this folder.
ARCHIVE_DIR = PROJECT_ROOT / "Project Files"
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from portable import repair_artifact_paths
print("Relocated saved references:", repair_artifact_paths(PROJECT_ROOT))

def missing_dataset_files():
    missing = []
    base = PROJECT_ROOT / "dataset" / "aptos2019"
    for filename, folder in (("train_1.csv", "train_images"), ("valid.csv", "val_images"), ("test.csv", "test_images")):
        csv_path = base / filename
        if not csv_path.is_file():
            missing.append(csv_path)
            continue
        with csv_path.open(encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                image = base / folder / (row["id_code"] + ".png")
                if not image.is_file():
                    missing.append(image)
    return missing

if missing_dataset_files():
    if not RESTORE_DATA_FROM_ARCHIVES:
        raise FileNotFoundError("Dataset is incomplete. Enable RESTORE_DATA_FROM_ARCHIVES or restore dataset/aptos2019 manually.")
    archives = sorted(ARCHIVE_DIR.glob("ITPG708Project-20260323T173934Z-3-*.zip"))
    if len(archives) != 5:
        raise FileNotFoundError(f"Expected all five project ZIP files under {ARCHIVE_DIR}. Update ARCHIVE_DIR if moved.")
    prefix = "ITPG708Project/dataset/aptos2019/"
    for archive in archives:
        print("Restoring dataset from", archive.name, flush=True)
        with zipfile.ZipFile(archive) as package:
            for entry in package.infolist():
                if not entry.filename.startswith(prefix) or entry.is_dir():
                    continue
                target = (PROJECT_ROOT / entry.filename.removeprefix("ITPG708Project/")).resolve()
                if not target.is_relative_to(PROJECT_ROOT):
                    raise ValueError("Archive entry escapes project folder")
                if target.is_file() and target.stat().st_size == entry.file_size:
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                partial = target.with_suffix(target.suffix + ".partial")
                with package.open(entry) as src, partial.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
                partial.replace(target)
    missing = missing_dataset_files()
    if missing:
        raise FileNotFoundError(f"Dataset still incomplete: {len(missing)} missing files; first: {missing[0]}")
print("Project ready:", PROJECT_ROOT)


## 1. Environment Setup


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

if 'PROJECT_ROOT' not in globals():
    raise RuntimeError('Run the Windows project setup cell first.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dr_pipeline import (
    load_project_config,
    clean_generated_outputs,
    notebook_prepare_data_overview,
    notebook_run_training,
    notebook_run_core_evaluation,
    notebook_run_xai,
    notebook_load_xai_committee_summary,
    notebook_load_xai_advanced_audit,
    notebook_run_visual_review,
    notebook_run_single_case_report,
)

display(Markdown(f'**Project root:** `{PROJECT_ROOT}`'))


## 2. Run Configuration


In [ ]:
CONFIG_PATH = str((PROJECT_ROOT / 'configs/base.yaml').resolve())
PIPELINE_BACKBONE = 'resnet50'
SEED = 1988
EVAL_SPLIT = 'test'
RUN_CLEAN_BEFORE_START = False
FORCE_RETRAIN = True  # Required: train a fresh model for the loss-function ablation.
ALLOW_TRAINING = True
XAI_SAFE_MODE = True
SHOW_ADVANCED_XAI_AUDIT = False
XAI_VIS_SAFE_MODE = True
XAI_SINGLE_SAFE_MODE = True

cfg = load_project_config(CONFIG_PATH)
cfg['training']['backbone'] = PIPELINE_BACKBONE
cfg['training']['loss_name'] = 'cross_entropy'  # Ablation: focal loss is replaced only by cross-entropy.
RUN_VERSION = 'ablation_cross_entropy_e15'
RUN_OUTPUT_ROOT = PROJECT_ROOT / 'artifacts' / 'runs' / f'{PIPELINE_BACKBONE}_{RUN_VERSION}'
cfg['paths'].update(artifacts_root=str(RUN_OUTPUT_ROOT), manifests_dir=str(RUN_OUTPUT_ROOT / 'manifests'), checkpoints_dir=str(RUN_OUTPUT_ROOT / 'checkpoints'), predictions_dir=str(RUN_OUTPUT_ROOT / 'predictions'), logs_dir=str(RUN_OUTPUT_ROOT / 'logs'), reports_root=str(RUN_OUTPUT_ROOT / 'reports'), figures_dir=str(RUN_OUTPUT_ROOT / 'reports' / 'figures'), tables_dir=str(RUN_OUTPUT_ROOT / 'reports' / 'tables'))
# Runtime-only overrides preserve the saved model's architecture and preprocessing.
cfg['training'].update(device='auto', num_workers=0, batch_size=4, epochs=15)
cfg['xai'].update(device='cpu', max_targets=4, shap_max_samples=4, shap_background_size=2, figure_dpi=240)
if RUN_CLEAN_BEFORE_START and not ALLOW_TRAINING:
    raise RuntimeError('Cleanup would remove the saved model. Keep cleanup disabled for the demo.')
protocol = str(cfg.get('data', {}).get('protocol', 'default'))

display(Markdown(f'**Config:** `{CONFIG_PATH}`'))
display(Markdown(f'**Output version:** `{RUN_VERSION}` | **Output folder:** `{RUN_OUTPUT_ROOT}`'))
display(Markdown(
    f'**Protocol:** `{protocol}` | **Seed:** `{SEED}` | **Split:** `{EVAL_SPLIT}` | '
    f'**Backbone:** `{cfg["training"].get("backbone", "resnet50")}`'
))

if protocol == 'benchmark':
    display(Markdown(
        f'**Benchmark split setup:** test_ratio={cfg["data"].get("profile_test_ratio", cfg["data"].get("test_ratio", 0.15)):.2f}, '
        f'val_within_train={cfg["data"].get("profile_val_ratio_within_train", cfg["data"].get("val_ratio", 0.10)):.2f}'
    ))
    display(Markdown(f'**Split seeds:** `{cfg["data"].get("split_seed_list", [SEED])}`'))
else:
    display(Markdown(
        f'**Configured split ratios:** train={cfg["data"]["train_ratio"]:.2f}, '
        f'test={cfg["data"]["test_ratio"]:.2f}, val={cfg["data"]["val_ratio"]:.2f}'
    ))

display(Markdown(
    f'**Preprocessing:** `{cfg.get("preprocessing", {}).get("enabled", True)}` | '
    f'**Loss:** `{cfg["training"].get("loss_name", "auto")}` | '
    f'**Weighted sampler:** `{cfg["training"].get("use_weighted_sampler", True)}` | '
    f'**Force retrain:** `{FORCE_RETRAIN}`'
))

display(Markdown('**Note:** If `RUN_CLEAN_BEFORE_START=True`, checkpoints are deleted first; retraining is expected even when `FORCE_RETRAIN=False`.'))


## 3. Optional Output Reset


In [ ]:
if RUN_CLEAN_BEFORE_START:
    clean_generated_outputs(CONFIG_PATH)
    print('Generated outputs cleaned.')
else:
    print('Cleanup skipped.')


## 4. Data Preparation


In [ ]:
data_overview = notebook_prepare_data_overview(cfg, seed=SEED, samples_per_split=3)
manifest_paths = data_overview['manifest_paths']
train_df = data_overview['train_df']
val_df = data_overview['val_df']
test_df = data_overview['test_df']

display(Markdown('### Manifest Files'))
display(pd.DataFrame([manifest_paths]).T.rename(columns={0: 'path'}))

display(Markdown('### Split Sizes'))
display(data_overview['split_summary'])

display(Markdown('### Overall Class Proportions'))
display(data_overview['overall_class_proportions'])

if len(data_overview['source_dataset_mix']):
    display(Markdown('### Source Dataset Mix (Roboflow vs APTOS)'))
    display(data_overview['source_dataset_mix'])
else:
    display(Markdown('### Source Dataset Mix'))
    display(Markdown('`source_dataset` column not found in manifests.'))

display(data_overview['class_count_fig'])
plt.close(data_overview['class_count_fig'])

display(data_overview['class_proportion_fig'])
plt.close(data_overview['class_proportion_fig'])

# LIME explainability configuration. LIME maps are generated in Section 7 after the model checkpoint is available.
lime_config = cfg.get('xai', {})
lime_settings = pd.DataFrame({
    'value': {
        'enabled': lime_config.get('lime_enabled', True),
        'num_samples': lime_config.get('lime_num_samples', 256),
        'batch_size': lime_config.get('lime_batch_size', 4),
        'superpixels': lime_config.get('lime_n_segments', 50),
    }
})
display(Markdown('### LIME Explanation Settings'))
display(lime_settings)
display(Markdown('_LIME highlights the image regions that locally support the model’s predicted diabetic-retinopathy class. The explanation figures are created in Section 7 after fine-tuning._'))


## 5. Model Fine-Tuning


In [ ]:
# Fail clearly rather than silently starting a 15-epoch training run.
from src.dr_pipeline import _checkpoint_config_signature, _find_matching_checkpoint_by_signature
if not ALLOW_TRAINING:
    if FORCE_RETRAIN:
        raise RuntimeError('FORCE_RETRAIN requires ALLOW_TRAINING=True.')
    match = _find_matching_checkpoint_by_signature(cfg, SEED, _checkpoint_config_signature(cfg))
    if match is None:
        raise RuntimeError('No readable matching checkpoint. Check dependencies and model configuration; training is disabled.')
training_out = notebook_run_training(
    cfg,
    seed=SEED,
    manifests=manifest_paths,
    force_retrain=FORCE_RETRAIN,
)
checkpoint_path = training_out['checkpoint_path']
RUN_ID = str(training_out['run_id'])

display(Markdown(f'Checkpoint path: `{checkpoint_path}`'))
display(Markdown(f'Run ID: `{RUN_ID}`'))
display(Markdown(
    f"**Checkpoint reuse:** `{training_out['reused_checkpoint']}` | "
    f"**Reason:** `{training_out['reuse_reason']}`"
))


## 6. Model Evaluation (Core)


In [ ]:
core_eval = notebook_run_core_evaluation(
    cfg,
    seed=SEED,
    split=EVAL_SPLIT,
)

display(Markdown(f'**Predictions used for evaluation:** `{core_eval["predictions_path"]}`'))
display(Markdown('### Evaluation Output Files'))
display(core_eval['eval_output_paths_df'])

display(Markdown('### Overall Metrics + Run Summary'))
display(core_eval['overall_summary_df'])

display(Markdown('### Per-Class Metrics'))
display(core_eval['per_class_df'])
